# SPORTPREDICT AI — Flujo de Entrenamiento Experimental (Google Colab)
Este cuaderno ejecuta el flujo reproducible de entrenamiento estadístico y Machine Learning (CPU estándar, sin requerir GPU) cumpliendo:
1. **Partición estrictamente cronológica** (`Entrenamiento -> Validación Temporal -> Prueba Final Retenida`).
2. **Cero fuga de información futura** (*Point-in-Time Feature Engineering*).
3. **Modelos A - E:** Poisson/Dixon-Coles, Elo Dinámico, Random Forest Calibrado, XGBoost Calibrado y Ensamble Ponderado.
4. **Exportación de artefacto `.joblib`** compatible con el backend FastAPI (`backend/artifacts/models/`).

In [ ]:
# 1. Instalación de dependencias abiertas (CPU)
!pip install -q numpy pandas scipy scikit-learn xgboost joblib

In [ ]:
import json
from datetime import datetime, timezone
import joblib
import numpy as np
import pandas as pd
from scipy.stats import poisson
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, log_loss
from xgboost import XGBClassifier

print('Entorno científico cargado correctamente.')

In [ ]:
# 2. Carga de CSV autorizado y validación de estructura
# En Google Colab puedes subir tu CSV autorizado o clonar el repositorio de SPORTPREDICT AI.
def validate_and_sort_chronologically(df: pd.DataFrame) -> pd.DataFrame:
    required = {'external_ref', 'kickoff_utc', 'home_team', 'away_team', 'home_goals', 'away_goals'}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f'Faltan columnas obligatorias: {missing}')
    df = df.drop_duplicates(subset=['external_ref']).dropna(subset=['home_goals', 'away_goals']).copy()
    df['kickoff_utc'] = pd.to_datetime(df['kickoff_utc'], utc=True)
    return df.sort_values('kickoff_utc').reset_index(drop=True)

print('Función de validación cronológica lista.')